# Fair-RAG Experiment Control Center

This notebook provides two core functions:
- summarize past experiments
- run new experiments with configurable knobs

Primary knobs:
- model
- max queries
- n_samples (stochastic runs)
- top-k
- print interval

In [6]:
from pathlib import Path
from datetime import datetime
import importlib
import json
import shutil

import pandas as pd
from IPython.display import display
from huggingface_hub import snapshot_download

import notebook_experiment_utils as neu
from utils import models_info

importlib.reload(neu)

ExperimentConfig = neu.ExperimentConfig
find_repo_root = neu.find_repo_root
resolve_python_executable = neu.resolve_python_executable
ensure_paths_exist = neu.ensure_paths_exist
run_retriever_grid = neu.run_retriever_grid
run_deterministic_reference = neu.run_deterministic_reference
run_mmr_deterministic = neu.run_mmr_deterministic
run_gold_deterministic_reference = neu.run_gold_deterministic_reference
run_gold_baseline = neu.run_gold_baseline
normalize_eu_grid = neu.normalize_eu_grid
raw_results_path = neu.raw_results_path

In [7]:
ROOT = find_repo_root()
PY = resolve_python_executable(ROOT)
ensure_paths_exist([PY])

print('Root   :', ROOT)
print('Python :', PY)
print('Models :', sorted(models_info.keys()))

Root   : /Users/asimk/Code/Fair-RAG
Python : /Users/asimk/Code/Fair-RAG/.venv/bin/python
Models : ['bonsai8BMLX1bit', 'flanT5Base', 'flanT5Small', 'flanT5XXL', 'lfm25MLX12B4bit', 'qwen35MLX4B4bit']


## Function 1: Summarize Past Experiments

In [3]:
def _mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else float('nan')

def _summarize_output_file(output_fp: Path) -> dict:
    if not output_fp.exists():
        return {
            'queries': 0,
            'mean_ee_d': float('nan'),
            'mean_ee_r': float('nan'),
            'metric_name': None,
            'mean_metric': float('nan'),
        }

    with output_fp.open('r', encoding='utf-8') as f:
        data = json.load(f)

    if not isinstance(data, dict) or not data:
        return {
            'queries': 0,
            'mean_ee_d': float('nan'),
            'mean_ee_r': float('nan'),
            'metric_name': None,
            'mean_metric': float('nan'),
        }

    ee_d_vals, ee_r_vals, metric_vals = [], [], []
    metric_name = None
    for payload in data.values():
        if not isinstance(payload, dict):
            continue
        ee = payload.get('EE', {}) or {}
        eu = payload.get('EU', {}) or {}
        if 'disparity' in ee:
            ee_d_vals.append(float(ee['disparity']))
        if 'relevance' in ee:
            ee_r_vals.append(float(ee['relevance']))
        if isinstance(eu, dict) and eu:
            metric_name = next(iter(eu.keys()))
            metric_vals.append(float(next(iter(eu.values()))))

    return {
        'queries': len(data),
        'mean_ee_d': _mean(ee_d_vals),
        'mean_ee_r': _mean(ee_r_vals),
        'metric_name': metric_name,
        'mean_metric': _mean(metric_vals),
    }

def summarize_past_experiments(
    generator_name: str | None = None,
    lamp_num: int | None = None,
    retriever_name: str | None = None,
    max_rows: int = 300,
    include_missing_outputs: bool = False,
    ) -> pd.DataFrame:
    runs_dir = ROOT / 'experiment_results' / 'runs'
    if not runs_dir.exists():
        print('No experiment_results/runs directory found.')
        return pd.DataFrame()

    rows = []
    for params_fp in runs_dir.glob('*/params.json'):
        with params_fp.open('r', encoding='utf-8') as f:
            params = json.load(f)

        gen = params.get('generator')
        lamp = params.get('lamp_num')
        retr = params.get('retriever')

        if generator_name is not None and gen != generator_name:
            continue
        if lamp_num is not None and lamp != lamp_num:
            continue
        if retriever_name is not None and retr != retriever_name:
            continue

        output_file = params.get('output_file')
        output_fp = Path(output_file) if output_file else Path()
        metric_summary = _summarize_output_file(output_fp) if output_file else {
            'queries': 0,
            'mean_ee_d': float('nan'),
            'mean_ee_r': float('nan'),
            'metric_name': None,
            'mean_metric': float('nan'),
        }

        if (not include_missing_outputs) and metric_summary['queries'] == 0:
            continue

        rows.append({
            'started_at': params.get('started_at'),
            'generator': gen,
            'lamp_num': lamp,
            'retriever': retr,
            'base_retriever': params.get('base_retriever'),
            'alpha': params.get('alpha'),
            'deterministic': params.get('deterministic'),
            'mmr_lambda': params.get('mmr_lambda'),
            'k': params.get('k'),
            'n_samples': params.get('n_samples'),
            'max_queries': params.get('max_queries'),
            'print_interval': params.get('print_interval'),
            'run_tag': params.get('run_tag'),
            'output_suffix': params.get('output_suffix'),
            'queries': metric_summary['queries'],
            'mean_ee_d': metric_summary['mean_ee_d'],
            'mean_ee_r': metric_summary['mean_ee_r'],
            'metric_name': metric_summary['metric_name'],
            'mean_metric': metric_summary['mean_metric'],
            'output_file': output_file,
            'run_dir': str(params_fp.parent.relative_to(ROOT)),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        print('No matching runs found.')
        return df

    df = df.sort_values('started_at', ascending=False).head(max_rows)
    display(df)
    return df

In [12]:
# Example summaries
_ = summarize_past_experiments()
# _ = summarize_past_experiments(generator_name='lfm25MLX12B4bit', lamp_num=4)
# _ = summarize_past_experiments(retriever_name='mmr')

,started_at,generator,lamp_num,retriever,base_retriever,alpha,deterministic,mmr_lambda,k,n_samples,...,print_interval,run_tag,output_suffix,queries,mean_ee_d,mean_ee_r,metric_name,mean_metric,output_file,run_dir
12,2026-04-04T02:26:57,lfm25MLX12B4bit,4,bm25,bm25,8,False,NaN,5,12,...,None,control_lfm25MLX12B4bit_bm25,,833,0.958133,0.315048,rouge-l,0.052768,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260404_022657_bm25_a...
36,2026-04-04T01:26:14,lfm25MLX12B4bit,4,bm25,bm25,4,False,NaN,5,12,...,None,control_lfm25MLX12B4bit_bm25,,833,0.708750,0.310704,rouge-l,0.052595,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260404_012614_bm25_a...
11,2026-04-04T00:20:43,lfm25MLX12B4bit,4,bm25,bm25,2,False,NaN,5,12,...,None,control_lfm25MLX12B4bit_bm25,,833,0.241213,0.288544,rouge-l,0.045223,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260404_002043_bm25_a...
2,2026-04-03T23:15:55,lfm25MLX12B4bit,4,bm25,bm25,1,False,NaN,5,12,...,None,control_lfm25MLX12B4bit_bm25,,833,0.193321,0.284190,rouge-l,0.043481,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260403_231555_bm25_a...
47,2026-04-03T23:10:31,lfm25MLX12B4bit,4,gold,gold,1,True,NaN,5,1,...,None,control_lfm25MLX12B4bit_bm25,_gold_deterministic,833,1.000000,0.983071,rouge-l,0.052485,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260403_231031_gold_a...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51,2026-03-29T15:58:00,flanT5Small,4,bm25,bm25,1,False,NaN,5,10,...,None,,_sanity,12,0.188667,0.205447,rouge-l,0.021648,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260329_155800_bm25_a...
3,2026-03-29T15:56:06,flanT5Small,4,gold,gold,8,False,NaN,5,10,...,None,,_sanity,12,0.513000,0.990238,rouge-l,0.054106,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260329_155606_gold_a...
6,2026-03-29T15:53:00,flanT5Small,4,gold,gold,8,False,NaN,5,10,...,None,,_sanity,12,0.513000,0.990238,rouge-l,0.054106,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260329_155300_gold_a...
61,2026-03-29T15:50:50,flanT5Small,4,gold,gold,8,False,NaN,5,10,...,None,,_sanity,12,0.513000,0.990238,rouge-l,0.054106,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260329_155050_gold_a...


In [5]:
_ = summarize_past_experiments(generator_name='flanT5Small', lamp_num=4)
# show only with queries=833, order by alpha, show only relevant columns like alpha, mean_ee_d, mean_ee_r, metric_name, mean_metric
_[_['queries'] == 833].sort_values('alpha')[['alpha', 'mean_ee_d', 'mean_ee_r', 'metric_name', 'mean_metric']]

,started_at,generator,lamp_num,retriever,base_retriever,alpha,deterministic,mmr_lambda,k,n_samples,...,print_interval,run_tag,output_suffix,queries,mean_ee_d,mean_ee_r,metric_name,mean_metric,output_file,run_dir
24,2026-03-30T12:43:13,flanT5Small,4,gold,gold,1,True,NaN,5,1,...,None,balanced,_gold_deterministic,833,1.000000,0.983071,rouge-l,0.074103,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260330_124313_gold_a...
7,2026-03-30T12:39:08,flanT5Small,4,gold,gold,1,True,NaN,5,12,...,None,balanced,_gold_deterministic,833,1.000000,0.983071,rouge-l,0.074103,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260330_123908_gold_a...
17,2026-03-30T12:28:43,flanT5Small,4,gold,gold,1,True,NaN,5,1,...,None,balanced,_gold_deterministic,833,1.000000,0.983071,rouge-l,0.074103,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260330_122843_gold_a...
19,2026-03-30T12:28:24,flanT5Small,4,bm25,bm25,8,False,NaN,5,10,...,None,,_sanity,12,0.977333,0.156738,rouge-l,0.023437,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260330_122824_bm25_a...
14,2026-03-30T12:26:53,flanT5Small,4,bm25,bm25,4,False,NaN,5,10,...,None,,_sanity,12,0.718667,0.166157,rouge-l,0.031739,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260330_122653_bm25_a...
21,2026-03-30T12:25:14,flanT5Small,4,bm25,bm25,2,False,NaN,5,10,...,None,,_sanity,12,0.240000,0.198781,rouge-l,0.020350,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260330_122514_bm25_a...
25,2026-03-30T12:23:36,flanT5Small,4,bm25,bm25,1,False,NaN,5,10,...,None,,_sanity,12,0.188667,0.205447,rouge-l,0.021648,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260330_122336_bm25_a...
30,2026-03-30T12:21:47,flanT5Small,4,gold,gold,8,False,NaN,5,10,...,None,,_sanity,12,0.513000,0.990238,rouge-l,0.054106,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260330_122147_gold_a...
28,2026-03-29T21:40:16,flanT5Small,4,mmr,gold,1,True,0.6,5,1,...,None,balanced,_mmr_on_gold_deterministic,833,1.000000,0.983071,rouge-l,0.074921,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260329_214016_gold_a...
32,2026-03-29T21:30:29,flanT5Small,4,mmr,gold,1,True,0.6,5,1,...,None,balanced,_mmr_on_gold_deterministic,833,1.000000,0.983071,rouge-l,0.074921,/Users/asimk/Code/Fair-RAG/experiment_results/...,experiment_results/runs/20260329_213029_gold_a...


,alpha,mean_ee_d,mean_ee_r,metric_name,mean_metric
24,1,1.000000,0.983071,rouge-l,0.074103
7,1,1.000000,0.983071,rouge-l,0.074103
17,1,1.000000,0.983071,rouge-l,0.074103
28,1,1.000000,0.983071,rouge-l,0.074921
32,1,1.000000,0.983071,rouge-l,0.074921
6,1,1.000000,0.983071,rouge-l,0.074921
12,1,1.000000,0.983071,rouge-l,0.074921
1,1,1.000000,0.983071,rouge-l,0.074921
8,1,1.000000,0.983071,rouge-l,0.074921
34,1,1.000000,0.983071,rouge-l,0.074921


In [ ]:
# Example filter: runs with at least 30 evaluated queries, sorted by utility metric
# Show only(!) relevant columns like 'generator', 'lamp_num', 'retriever', 'mean_metric', 'queries'

_[_['queries'] >= 30][['generator', 'retriever','alpha','lamp_num', 'mean_metric', 'mean_ee_r', 'mean_ee_d',
            'queries']].sort_values([ 'mean_metric'], ascending=[False]).head(30)

,generator,lamp_num,mean_metric,mean_ee_r,mean_ee_d,queries
27,flanT5Base,4,0.082620,0.994427,1.000000,30
8,flanT5Small,4,0.074921,0.983071,1.000000,833
60,flanT5Small,4,0.074921,0.983071,1.000000,833
62,flanT5Small,4,0.074921,0.983071,1.000000,833
14,flanT5Small,4,0.074921,0.983071,1.000000,833
1,flanT5Small,4,0.074921,0.983071,1.000000,833
22,flanT5Small,4,0.074921,0.983071,1.000000,833
55,flanT5Small,4,0.074921,0.983071,1.000000,833
13,flanT5Small,4,0.074103,0.983071,1.000000,833
29,flanT5Small,4,0.074103,0.983071,1.000000,833


## Function 2: Run a New Experiment

In [8]:
def resolve_model_repo(model: str) -> str:
    info = models_info.get(model, {})
    return info.get('model_id', model)

def ensure_mlx_model_cached(model: str) -> Path:
    model_repo = resolve_model_repo(model)
    cache_path = Path(snapshot_download(model_repo, local_files_only=True))
    print('MLX model cache:', cache_path)
    return cache_path

def prepare_model_for_warning_free_run(model: str, allow_hf_download_if_missing: bool = False) -> None:
    if model not in models_info:
        raise ValueError(f'Unknown model: {model}. Available: {sorted(models_info)}')

    backend = models_info[model].get('backend', 'hf')
    if backend != 'mlx':
        return

    model_repo = resolve_model_repo(model)
    try:
        cache_path = ensure_mlx_model_cached(model)
        print(f'[ok] using local cache for {model_repo}')
        print(f'[ok] no re-download path: {cache_path}')
    except Exception:
        if not allow_hf_download_if_missing:
            raise FileNotFoundError(
                f'Model {model_repo} is not in local cache. '
                'For stable runs without important warnings, pre-download once before running.'
            )
        cache_path = Path(snapshot_download(model_repo, local_files_only=False))
        print(f'[downloaded-once] cached at: {cache_path}')

def validate_run_preconditions(model: str, lamp_num: int, retriever_name: str) -> None:
    problems: list[str] = []

    data_dir = ROOT / 'data' / f'lamp_utility_labels_{model}'
    required = [
        data_dir / f'{lamp_num}_user_dev_inputs.json',
        data_dir / f'{lamp_num}_user_dev_outputs.json',
        data_dir / f'{lamp_num}_relevance_mapping.tsv',
    ]
    for fp in required:
        if not fp.exists():
            problems.append(f'missing data file: {fp}')

    retrieval_fp = ROOT / 'retrieval' / 'retrieval_results' / model / retriever_name / f'{lamp_num}.json'
    if retriever_name in {'splade', 'mmr'} and not retrieval_fp.exists():
        problems.append(
            f'missing retrieval file for {retriever_name}: {retrieval_fp} (build retrieval first to avoid runtime warnings/failures)'
        )

    if problems:
        msg = 'Precheck failed. Resolve these before running:\n- ' + '\n- '.join(problems)
        raise RuntimeError(msg)

    print('[ok] precheck passed')

In [9]:
def ensure_generator_layout(generator_name: str, lamp_num: int = 4, source_generator: str = 'flanT5Small', copy_bm25_retrieval: bool = True) -> None:
    source_data = ROOT / 'data' / f'lamp_utility_labels_{source_generator}'
    target_data = ROOT / 'data' / f'lamp_utility_labels_{generator_name}'
    target_data.mkdir(parents=True, exist_ok=True)

    shared = [
        f'{lamp_num}_user_dev_inputs.json',
        f'{lamp_num}_user_dev_outputs.json',
        f'{lamp_num}_relevance_mapping.tsv',
    ]
    for name in shared:
        src = source_data / name
        dst = target_data / name
        if not dst.exists() and src.exists():
            shutil.copy2(src, dst)
            print('copied', dst)

    if copy_bm25_retrieval:
        src = ROOT / 'retrieval' / 'retrieval_results' / source_generator / 'bm25' / f'{lamp_num}.json'
        dst_dir = ROOT / 'retrieval' / 'retrieval_results' / generator_name / 'bm25'
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / f'{lamp_num}.json'
        if not dst.exists() and src.exists():
            shutil.copy2(src, dst)
            print('copied', dst)

def build_experiment_config(
    model: str,
    lamp_num: int,
    max_queries: int | None,
    n_samples: int,
    top_k: int,
    print_interval: int,
    retriever_name: str = 'bm25',
    alphas: tuple[int, ...] = (1, 2, 4, 8),
    run_tag: str = 'custom',
    skip_existing: bool = True,
    save_llm_outputs: bool = True,
) -> ExperimentConfig:
    return ExperimentConfig(
        root=ROOT,
        python_exe=PY,
        generator_name=model,
        lamp_num=lamp_num,
        retriever_name=retriever_name,
        alphas=alphas,
        max_queries=max_queries,
        n_samples=n_samples,
        k=top_k,
        remove_temp_files=True,
        skip_existing=skip_existing,
        mmr_base_retriever='bm25',
        mmr_lambda=0.6,
        run_tag=run_tag,
        print_interval=print_interval,
        save_llm_outputs=save_llm_outputs,
    )


In [10]:
def validate_prism_mlx_toolchain_if_needed(model: str) -> None:
    backend = models_info.get(model, {}).get('backend', 'hf')
    if backend != 'mlx':
        return

    model_repo = resolve_model_repo(model).lower()
    if 'prism-ml/bonsai' not in model_repo:
        return

    import shutil
    if shutil.which('metal') is None:
        raise RuntimeError(
            'Prism Bonsai requires Prism MLX fork build support, but metal compiler is missing. '
            'Install full Xcode and switch developer dir:\n'
            '  sudo xcode-select -s /Applications/Xcode.app/Contents/Developer\n'
            'Then reinstall:\n'
            '  pip install mlx-lm\n'
            '  pip install "mlx @ git+https://github.com/PrismML-Eng/mlx.git@prism"'
        )

def validate_run_preconditions(model: str, lamp_num: int, retriever_name: str) -> None:
    problems: list[str] = []

    validate_prism_mlx_toolchain_if_needed(model)

    data_dir = ROOT / 'data' / f'lamp_utility_labels_{model}'
    required = [
        data_dir / f'{lamp_num}_user_dev_inputs.json',
        data_dir / f'{lamp_num}_user_dev_outputs.json',
        data_dir / f'{lamp_num}_relevance_mapping.tsv',
    ]
    for fp in required:
        if not fp.exists():
            problems.append(f'missing data file: {fp}')

    retrieval_fp = ROOT / 'retrieval' / 'retrieval_results' / model / retriever_name / f'{lamp_num}.json'
    if retriever_name in {'splade', 'mmr'} and not retrieval_fp.exists():
        problems.append(
            f'missing retrieval file for {retriever_name}: {retrieval_fp} (build retrieval first to avoid runtime warnings/failures)'
        )

    if problems:
        msg = 'Precheck failed. Resolve these before running:\n- ' + '\n- '.join(problems)
        raise RuntimeError(msg)

    print('[ok] precheck passed')

def run_new_experiment(
    model: str,
    lamp_num: int,
    max_queries: int | None,
    n_samples: int,
    top_k: int,
    print_interval: int,
    retriever_name: str = 'bm25',
    run_grid: bool = True,
    run_deterministic: bool = True,
    run_mmr_deterministic_flag: bool = False,
    run_gold: bool = False,
    run_gold_deterministic: bool = False,
    run_normalize: bool = True,
    allow_hf_download_if_missing: bool = False,
    strict_precheck: bool = True,
    skip_existing: bool = True,
    save_llm_outputs: bool = True,
) -> ExperimentConfig:
    if model not in models_info:
        raise ValueError(f'Unknown model: {model}. Available: {sorted(models_info)}')

    prepare_model_for_warning_free_run(
        model=model,
        allow_hf_download_if_missing=allow_hf_download_if_missing,
    )

    cfg = build_experiment_config(
        model=model,
        lamp_num=lamp_num,
        max_queries=max_queries,
        n_samples=n_samples,
        top_k=top_k,
        print_interval=print_interval,
        retriever_name=retriever_name,
        run_tag=f'control_{model}_{retriever_name}_lamp{lamp_num}',
        skip_existing=skip_existing,
        save_llm_outputs=save_llm_outputs,
    )

    if model in models_info and models_info[model].get('backend') == 'mlx':
        models_info[model].setdefault('model_kwargs', {})
        models_info[model]['model_kwargs']['allow_hf_download'] = allow_hf_download_if_missing

    ensure_generator_layout(generator_name=model, lamp_num=cfg.lamp_num, source_generator='flanT5Small', copy_bm25_retrieval=(retriever_name == 'bm25'))

    if strict_precheck:
        validate_run_preconditions(model=model, lamp_num=cfg.lamp_num, retriever_name=retriever_name)

    print('Running config:', cfg)

    if run_gold:
        run_gold_baseline(cfg, alpha=8)
    if run_deterministic:
        run_deterministic_reference(cfg, alpha=1, output_suffix='_deterministic')
    if run_mmr_deterministic_flag:
        run_mmr_deterministic(cfg, alpha=1, output_suffix='_mmr_deterministic')
    if run_gold_deterministic:
        run_gold_deterministic_reference(cfg, alpha=1, output_suffix='_gold_deterministic')
    if run_grid:
        run_retriever_grid(cfg)
    if run_normalize:
        normalize_eu_grid(cfg, normalization_scope='all-settings')

    return cfg


In [12]:
# Requested run matrix: LaMP2 + flanT5Small
MODEL = 'flanT5Small'
LAMP_NUM = 2
MAX_QUERIES = None      # None => all LaMP2 queries
N_SAMPLES = 10          # stochastic runs
TOP_K = 5
PRINT_INTERVAL = 25
RETRIEVER = 'bm25'      # base retriever for deterministic + stochastic runs

# Keep False for warning-free/no-re-download behavior on MLX models.
ALLOW_HF_DOWNLOAD_IF_MISSING = False

# If True, block launch when required files or toolchain pieces are missing.
STRICT_PRECHECK = True

# Set False to force a full rerun even if output files already exist.
SKIP_EXISTING = False

# Keep raw model generations in run logs for later cleaning/debugging.
SAVE_LLM_OUTPUTS = True

# Run matrix
RUN_GRID = True                    # stochastic bm25: alpha 1,2,4,8
RUN_DETERMINISTIC = True           # deterministic bm25
RUN_MMR_DETERMINISTIC = True       # deterministic mmr+bm25
RUN_GOLD = False                   # user asked deterministic gold only
RUN_GOLD_DETERMINISTIC = True      # deterministic gold
RUN_NORMALIZE = True
RUN_NOW = True

if RUN_NOW:
    cfg_used = run_new_experiment(
        model=MODEL,
        lamp_num=LAMP_NUM,
        max_queries=MAX_QUERIES,
        n_samples=N_SAMPLES,
        top_k=TOP_K,
        print_interval=PRINT_INTERVAL,
        retriever_name=RETRIEVER,
        run_grid=RUN_GRID,
        run_deterministic=RUN_DETERMINISTIC,
        run_mmr_deterministic_flag=RUN_MMR_DETERMINISTIC,
        run_gold=RUN_GOLD,
        run_gold_deterministic=RUN_GOLD_DETERMINISTIC,
        run_normalize=RUN_NORMALIZE,
        allow_hf_download_if_missing=ALLOW_HF_DOWNLOAD_IF_MISSING,
        strict_precheck=STRICT_PRECHECK,
        skip_existing=SKIP_EXISTING,
        save_llm_outputs=SAVE_LLM_OUTPUTS,
    )
    print('Finished run for model:', cfg_used.generator_name, '| lamp:', cfg_used.lamp_num)
else:
    print('Set RUN_NOW = True to launch the experiment.')


[ok] precheck passed
Running config: ExperimentConfig(root=PosixPath('/Users/asimk/Code/Fair-RAG'), python_exe=PosixPath('/Users/asimk/Code/Fair-RAG/.venv/bin/python'), generator_name='flanT5Small', lamp_num=2, retriever_name='bm25', alphas=(1, 2, 4, 8), max_queries=None, n_samples=10, k=5, remove_temp_files=True, skip_existing=False, mmr_base_retriever='bm25', mmr_lambda=0.6, run_tag='control_flanT5Small_bm25_lamp2', print_interval=25, save_llm_outputs=True)

> /Users/asimk/Code/Fair-RAG/.venv/bin/python experiment.py --generator_name flanT5Small --lamp_num 2 --retriever_name bm25 --alpha 1 --k 5 --n_samples 1 --deterministic_ranking --output_suffix _deterministic --remove_temp_files --run_tag control_flanT5Small_bm25_lamp2 --print_interval 25 --save_llm_outputs
[run-log] /Users/asimk/Code/Fair-RAG/experiment_results/runs/20260406_140914_bm25_alpha1_deterministic_flanT5Small_lamp2_nqall_control_flanT5Small_bm25_lamp2
[run-log] llm outputs will be saved to /Users/asimk/Code/Fair-RAG/ex

In [13]:
# Additional targeted run: deterministic MMR with lambda=0.35
MMR_LAMBDA_EXTRA = 0.35
MMR_SUFFIX_EXTRA = '_mmr_deterministic_lambda035'

cfg_mmr_lambda035 = build_experiment_config(
    model=MODEL,
    lamp_num=LAMP_NUM,
    max_queries=MAX_QUERIES,
    n_samples=N_SAMPLES,
    top_k=TOP_K,
    print_interval=PRINT_INTERVAL,
    retriever_name=RETRIEVER,
    run_tag=f'control_{MODEL}_{RETRIEVER}_lamp{LAMP_NUM}_mmr035',
    skip_existing=SKIP_EXISTING,
    save_llm_outputs=SAVE_LLM_OUTPUTS,
    alphas=(1,),
)
cfg_mmr_lambda035.mmr_lambda = MMR_LAMBDA_EXTRA

ensure_generator_layout(
    generator_name=MODEL,
    lamp_num=LAMP_NUM,
    source_generator='flanT5Small',
    copy_bm25_retrieval=True,
  )
validate_run_preconditions(model=MODEL, lamp_num=LAMP_NUM, retriever_name='bm25')

print('Running extra deterministic MMR config:', cfg_mmr_lambda035)
run_mmr_deterministic(
    cfg_mmr_lambda035,
    alpha=1,
    output_suffix=MMR_SUFFIX_EXTRA,
)
print('Finished extra deterministic MMR run with lambda =', MMR_LAMBDA_EXTRA)

[ok] precheck passed
Running extra deterministic MMR config: ExperimentConfig(root=PosixPath('/Users/asimk/Code/Fair-RAG'), python_exe=PosixPath('/Users/asimk/Code/Fair-RAG/.venv/bin/python'), generator_name='flanT5Small', lamp_num=2, retriever_name='bm25', alphas=(1,), max_queries=None, n_samples=10, k=5, remove_temp_files=True, skip_existing=False, mmr_base_retriever='bm25', mmr_lambda=0.35, run_tag='control_flanT5Small_bm25_lamp2_mmr035', print_interval=25, save_llm_outputs=True)

> /Users/asimk/Code/Fair-RAG/.venv/bin/python experiment.py --generator_name flanT5Small --lamp_num 2 --retriever_name mmr --alpha 1 --k 5 --n_samples 1 --deterministic_ranking --mmr_base_retriever bm25 --mmr_lambda 0.35 --output_suffix _mmr_deterministic_lambda035 --remove_temp_files --run_tag control_flanT5Small_bm25_lamp2_mmr035 --print_interval 25 --save_llm_outputs
[run-log] /Users/asimk/Code/Fair-RAG/experiment_results/runs/20260406_161118_bm25_alpha1_mmr-deterministic-lambda035_flanT5Small_lamp2_nqa

In [ ]:
# Full T5Base run: same LaMP2 matrix plus extra deterministic MMR lambda=0.35
T5BASE_MODEL = 'flanT5Base'
T5BASE_LAMP_NUM = 2
T5BASE_MAX_QUERIES = None
T5BASE_N_SAMPLES = 10
T5BASE_TOP_K = 5
T5BASE_PRINT_INTERVAL = 25
T5BASE_RETRIEVER = 'bm25'
T5BASE_ALLOW_HF_DOWNLOAD_IF_MISSING = False
T5BASE_STRICT_PRECHECK = True
T5BASE_SKIP_EXISTING = False
T5BASE_SAVE_LLM_OUTPUTS = True

cfg_t5base = run_new_experiment(
    model=T5BASE_MODEL,
    lamp_num=T5BASE_LAMP_NUM,
    max_queries=T5BASE_MAX_QUERIES,
    n_samples=T5BASE_N_SAMPLES,
    top_k=T5BASE_TOP_K,
    print_interval=T5BASE_PRINT_INTERVAL,
    retriever_name=T5BASE_RETRIEVER,
    run_grid=True,
    run_deterministic=True,
    run_mmr_deterministic_flag=True,
    run_gold=False,
    run_gold_deterministic=True,
    run_normalize=True,
    allow_hf_download_if_missing=T5BASE_ALLOW_HF_DOWNLOAD_IF_MISSING,
    strict_precheck=T5BASE_STRICT_PRECHECK,
    skip_existing=T5BASE_SKIP_EXISTING,
    save_llm_outputs=T5BASE_SAVE_LLM_OUTPUTS,
 )
print('Finished full run for model:', cfg_t5base.generator_name, '| lamp:', cfg_t5base.lamp_num)

T5BASE_MMR_LAMBDA_EXTRA = 0.35
T5BASE_MMR_SUFFIX_EXTRA = '_mmr_deterministic_lambda035'

cfg_t5base_mmr035 = build_experiment_config(
    model=T5BASE_MODEL,
    lamp_num=T5BASE_LAMP_NUM,
    max_queries=T5BASE_MAX_QUERIES,
    n_samples=T5BASE_N_SAMPLES,
    top_k=T5BASE_TOP_K,
    print_interval=T5BASE_PRINT_INTERVAL,
    retriever_name=T5BASE_RETRIEVER,
    run_tag=f'control_{T5BASE_MODEL}_{T5BASE_RETRIEVER}_lamp{T5BASE_LAMP_NUM}_mmr035',
    skip_existing=T5BASE_SKIP_EXISTING,
    save_llm_outputs=T5BASE_SAVE_LLM_OUTPUTS,
    alphas=(1,),
)
cfg_t5base_mmr035.mmr_lambda = T5BASE_MMR_LAMBDA_EXTRA

ensure_generator_layout(
    generator_name=T5BASE_MODEL,
    lamp_num=T5BASE_LAMP_NUM,
    source_generator='flanT5Small',
    copy_bm25_retrieval=True,
 )
validate_run_preconditions(model=T5BASE_MODEL, lamp_num=T5BASE_LAMP_NUM, retriever_name='bm25')

print('Running extra deterministic MMR config:', cfg_t5base_mmr035)
run_mmr_deterministic(
    cfg_t5base_mmr035,
    alpha=1,
    output_suffix=T5BASE_MMR_SUFFIX_EXTRA,
 )
print('Finished extra deterministic MMR run for model:', T5BASE_MODEL, '| lambda =', T5BASE_MMR_LAMBDA_EXTRA)

In [17]:
import importlib, json
import numpy as np
import pandas as pd
from IPython.display import display
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

import notebook_experiment_utils as neu
importlib.reload(neu)

from utils import models_info, trim_sentence_by_token_len
from data.lamp_handler import LaMPHandler
from eval.lamp_metrics import get_metric_fn_rouge_L
from generator.lm import PromptLM
from generator.lm_mlx import PromptLMMLX

# ── Diagnostic knobs ──────────────────────────────────────────────────────────
DIAG_MODELS   = ['flanT5Small', 'lfm25MLX12B4bit', 'qwen35MLX4B4bit']
DIAG_LAMP_NUM = 4
DIAG_N        = 10    # first N queries
DIAG_K        = 5     # top-k profiles from BM25

# ── Helpers ───────────────────────────────────────────────────────────────────
def _diag_resolve_tok(model_key):
    info = models_info[model_key]
    mid  = info['model_id']
    try:
        return str(snapshot_download(mid, local_files_only=True))
    except Exception:
        return str(snapshot_download(mid, local_files_only=False))

def _diag_build_gen(model_key):
    info = models_info[model_key]
    if info.get('backend') == 'mlx':
        return PromptLMMLX(model_name=model_key, model_kwargs=info.get('model_kwargs', {}))
    return PromptLM(model_name=model_key)

# ── Load shared data (flanT5Small layout is the canonical source) ─────────────
_data_dir = ROOT / 'data' / 'lamp_utility_labels_flanT5Small'
_rr_fp    = ROOT / 'retrieval' / 'retrieval_results' / 'flanT5Small' / 'bm25' / f'{DIAG_LAMP_NUM}.json'

with _rr_fp.open() as _f:
    _retrieval = json.load(_f)

with (_data_dir / f'{DIAG_LAMP_NUM}_user_dev_inputs.json').open() as _f:
    _all_inputs = json.load(_f)          # plain list of dicts

with (_data_dir / f'{DIAG_LAMP_NUM}_user_dev_outputs.json').open() as _f:
    _all_outputs = json.load(_f)['golds']   # list of {id, output}

_inputs_n  = _all_inputs[:DIAG_N]
_outputs_n = _all_outputs[:DIAG_N]

# base rows
_rows = [
    {'qid': inp['id'], 'query': inp['input'], 'gold': out['output']}
    for inp, out in zip(_inputs_n, _outputs_n)
]

_metric_fn = get_metric_fn_rouge_L()

# ── Run each model ────────────────────────────────────────────────────────────
for _model_key in DIAG_MODELS:
    print(f'\n--- {_model_key} ---', flush=True)
    _tok_src   = _diag_resolve_tok(_model_key)
    _tokenizer = AutoTokenizer.from_pretrained(_tok_src, local_files_only=False)
    _tok_max   = max(1, int(_tokenizer.model_max_length * 0.8))

    _handler = LaMPHandler(
        lamp_dir_name=f'lamp_utility_labels_{_model_key}',
        split_type='user',
        tokenizer_model_name=_tok_src,
        k=DIAG_K,
    )
    _aip  = _handler.get_aip_func(lamp_num=DIAG_LAMP_NUM)
    _gen  = _diag_build_gen(_model_key)
    _preds = []

    for _inp in _inputs_n:
        _qid    = _inp['id']
        _rr_qid = _retrieval.get(_qid, [])
        _scores = np.array([float(x[1]) for x in _rr_qid])
        _top_i  = np.argsort(-_scores, kind='mergesort')[:DIAG_K]
        _pids   = [_rr_qid[i][0] for i in _top_i]
        _profs  = _handler.find_profiles_by_pids(DIAG_LAMP_NUM, _qid, _pids)
        _prompt = _aip(question=_inp['input'], profiles=_profs)
        _prompt = trim_sentence_by_token_len(_prompt, tokenizer=_tokenizer, max_tok_len=_tok_max)
        _pred   = _gen.answer_question(final_prompt=_prompt).strip() or '<empty>'
        _preds.append(_pred)

    _scores_list = _metric_fn(_preds, [r['gold'] for r in _rows])
    for _i, (_pred, _s) in enumerate(zip(_preds, _scores_list)):
        _rows[_i][f'pred_{_model_key}'] = _pred
        _rows[_i][f'rl_{_model_key}']   = round(float(_s), 4)

    del _gen

# ── Display ───────────────────────────────────────────────────────────────────
def _trunc(t, n=100):
    t = str(t).replace('\n', ' ').strip()
    return t if len(t) <= n else t[:n] + '…'

_df = pd.DataFrame([
    {
        'qid':   r['qid'],
        'query': _trunc(r['query'], 110),
        'gold':  _trunc(r['gold'],  80),
        **{f'pred_{m}': _trunc(r[f'pred_{m}'], 80) for m in DIAG_MODELS},
        **{f'RL_{m}':   r[f'rl_{m}']           for m in DIAG_MODELS},
    }
    for r in _rows
])

display(_df)

print('\nMean rouge-L per model:')
for _m in DIAG_MODELS:
    print(f'  {_m:20s}  {_df[f"RL_{_m}"].mean():.4f}')


--- flanT5Small ---


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 9060.62it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



--- lfm25MLX12B4bit ---

--- qwen35MLX4B4bit ---


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 164482.51it/s]


,qid,query,gold,pred_flanT5Small,pred_lfm25MLX12B4bit,pred_qwen35MLX4B4bit,RL_flanT5Small,RL_lfm25MLX12B4bit,RL_qwen35MLX4B4bit
0,312,Generate a headline for the following article:...,Home Alone After School: Is Your Child Ready?,Parents need to be sure their children have th...,"**Headline:** **""Summer Safety Tips: Protect...",Thinking Process: 1. **Analyze the Request:*...,0.0000,0.2000,0.0449
1,313,Generate a headline for the following article:...,The Art of Parenting,What's the best way to raise your children?,"**Headline:** ""Raising Children: No One-Size...",Thinking Process: 1. **Analyze the Request:*...,0.0000,0.1176,0.0263
2,315,Generate a headline for the following article:...,Top City Officials Charged In Flint Water Cris...,Trump's attorney general picks Jeff Sessions W...,"**Headline:** ""Flint's Tragic Loss Highlights ...",Thinking Process: 1. **Analyze the Request:*...,0.0278,0.0000,0.0000
3,316,Generate a headline for the following article:...,Workers Fired After ‘Day Without Immigrants’ P...,Apple's North Star hasn't changed,"**Headline:** ""Michigan Workers Sue Over Tru...",Thinking Process: 1. **Analyze the Request:*...,0.0000,0.0800,0.0227
4,317,Generate a headline for the following article:...,Dutch Lawmakers Make ‘Historic’ Move To Expand...,A crash that killed five cyclists in Michigan ...,"""Parliament Approves Bill to Decriminalize Mar...",Thinking Process: 1. **Analyze the Request:*...,0.0000,0.0000,0.0000
5,318,Generate a headline for the following article:...,United Airlines CEO Somehow Won A Major PR Awa...,"'No one is above the law, not even those who w...","""Klein Blasts Congressman Over Government Shut...",Thinking Process: 1. **Analyze the Request:*...,0.0000,0.0000,0.0286
6,319,Generate a headline for the following article:...,Tea Party Lawmakers' Reported Affair Cover-Up ...,State Rep. Matt Rinaldi's call sparked an alte...,**Headline:** Texas Lawmaker Calls ICE On Grou...,Thinking Process: 1. **Analyze the Request:*...,0.0000,0.0000,0.0000
7,3110,Generate a headline for the following article:...,"ICE Officers Dined At Cafe, Then Arrested 3 Of...",ICE Arrests: 'My family members were picked up...,"""Immigration Enforcement Targets: Officials Cl...",Thinking Process: 1. **Analyze the Request:*...,0.0690,0.0000,0.0000
8,3111,Generate a headline for the following article:...,It’s Time For Presidential Candidates To Talk ...,Activists Say Betsy DeVos' Nomination Puts Kid...,"**Headline:** ""Trump's Education Pick Faces Sc...",Thinking Process: 1. **Analyze the Request:*...,0.0455,0.0000,0.0250
9,3112,Generate a headline for the following article:...,"Detroit's Abandoned Ruins Are Captivating, But...",'Dedicate Your Vote To A Woman' is the title f...,"**Headline:** ""Renovating the Past: How Citi...",Thinking Process: 1. **Analyze the Request:*...,0.0000,0.0909,0.0000



Mean rouge-L per model:
  flanT5Small           0.0142
  lfm25MLX12B4bit       0.0488
  qwen35MLX4B4bit       0.0147


In [19]:
# Refresh MLX predictions with latest generator code, then print a cleaner comparison view.
import importlib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

import generator.lm_mlx as lm_mlx
importlib.reload(lm_mlx)
PromptLMMLX = lm_mlx.PromptLMMLX

if '_rows' not in globals() or not _rows:
    raise RuntimeError('No diagnostic rows found. Run the previous diagnostic cell first.')

def _pretty_trunc(text: str, n: int = 180) -> str:
    text = str(text).replace('\n', ' ').strip()
    return text if len(text) <= n else text[:n] + '...'

def _resolve_tok(model_key: str) -> str:
    mid = models_info[model_key]['model_id']
    try:
        return str(snapshot_download(mid, local_files_only=True))
    except Exception:
        return str(snapshot_download(mid, local_files_only=False))

# Recompute only MLX model outputs into _rows (keeps flan columns as-is).
for _m in [m for m in DIAG_MODELS if models_info.get(m, {}).get('backend') == 'mlx']:
    print(f'Refreshing {_m} with reloaded generator...')
    _tok_src = _resolve_tok(_m)
    _tokenizer = AutoTokenizer.from_pretrained(_tok_src, local_files_only=False)
    _tok_max = max(1, int(_tokenizer.model_max_length * 0.8))
    _handler = LaMPHandler(
        lamp_dir_name=f'lamp_utility_labels_{_m}',
        split_type='user',
        tokenizer_model_name=_tok_src,
        k=DIAG_K,
    )
    _aip = _handler.get_aip_func(lamp_num=DIAG_LAMP_NUM)
    _gen = PromptLMMLX(model_name=_m, model_kwargs=models_info[_m].get('model_kwargs', {}))

    _preds = []
    for _inp in _inputs_n:
        _qid = _inp['id']
        _rr_qid = _retrieval.get(_qid, [])
        _scores = np.array([float(x[1]) for x in _rr_qid])
        _top_i = np.argsort(-_scores, kind='mergesort')[:DIAG_K]
        _pids = [_rr_qid[i][0] for i in _top_i]
        _profs = _handler.find_profiles_by_pids(DIAG_LAMP_NUM, _qid, _pids)
        _prompt = _aip(question=_inp['input'], profiles=_profs)
        _prompt = trim_sentence_by_token_len(_prompt, tokenizer=_tokenizer, max_tok_len=_tok_max)
        _pred = _gen.answer_question(final_prompt=_prompt).strip() or '<empty>'
        _preds.append(_pred)

    _scores_list = _metric_fn(_preds, [r['gold'] for r in _rows])
    for _i, (_pred, _s) in enumerate(zip(_preds, _scores_list)):
        _rows[_i][f'pred_{_m}'] = _pred
        _rows[_i][f'rl_{_m}'] = round(float(_s), 4)

# Compact table (easy scan)
compact_rows = []
for r in _rows:
    row = {
        'qid': r['qid'],
        'query': _pretty_trunc(r['query'], 120),
        'gold': _pretty_trunc(r['gold'], 90),
    }
    for m in DIAG_MODELS:
        row[f'{m}_RL'] = r.get(f'rl_{m}', float('nan'))
        row[f'{m}_pred'] = _pretty_trunc(r.get(f'pred_{m}', ''), 120)
    compact_rows.append(row)

df_compact = pd.DataFrame(compact_rows)
display(df_compact)

display(Markdown('### Mean Rouge-L by Model'))
summary = pd.DataFrame([
    {'model': m, 'mean_rouge_l': round(df_compact[f'{m}_RL'].mean(), 4)}
    for m in DIAG_MODELS
]).sort_values('mean_rouge_l', ascending=False)
display(summary)

# Full readable blocks
display(Markdown('### Query-by-Query Details'))
for i, r in enumerate(_rows, start=1):
    display(Markdown(f'#### Query {i} (qid={r["qid"]})'))
    print('Query :', r['query'])
    print('Gold  :', r['gold'])
    for m in DIAG_MODELS:
        print(f'[{m}] RL={r.get(f"rl_{m}", float("nan")):.4f}')
        print('  ', r.get(f'pred_{m}', ''))
    print('-' * 100)

Refreshing lfm25MLX12B4bit with reloaded generator...
Refreshing qwen35MLX4B4bit with reloaded generator...


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 192841.56it/s]


,qid,query,gold,flanT5Small_RL,flanT5Small_pred,lfm25MLX12B4bit_RL,lfm25MLX12B4bit_pred,qwen35MLX4B4bit_RL,qwen35MLX4B4bit_pred
0,312,Generate a headline for the following article:...,Home Alone After School: Is Your Child Ready?,0.0000,Parents need to be sure their children have th...,0.2105,Summer Safety Tips: Protecting Your Child from...,0.1176,"Staying Home Alone: Ensuring Safety, Maturity,..."
1,313,Generate a headline for the following article:...,The Art of Parenting,0.0000,What's the best way to raise your children?,0.1250,Raising Children: No One-Size-Fits-All Formula...,0.0000,There Is No Right Way to Raise Your Children
2,315,Generate a headline for the following article:...,Top City Officials Charged In Flint Water Cris...,0.0278,Trump's attorney general picks Jeff Sessions W...,0.0000,Flint's Tragic Loss Highlights Failures of Tru...,0.0870,Michigan AG Bill Schuette: Flint Was a Casualt...
3,316,Generate a headline for the following article:...,Workers Fired After ‘Day Without Immigrants’ P...,0.0000,Apple's North Star hasn't changed,0.0833,Michigan Workers Sue Over Trump’s Executive Or...,0.0000,Michigan Employees Take Case to National Labor...
4,317,Generate a headline for the following article:...,Dutch Lawmakers Make ‘Historic’ Move To Expand...,0.0000,A crash that killed five cyclists in Michigan ...,0.0000,Parliament Approves Bill to Decriminalize Mari...,0.0000,Canadian Parliament Clears Bill to Decriminali...
5,318,Generate a headline for the following article:...,United Airlines CEO Somehow Won A Major PR Awa...,0.0000,"'No one is above the law, not even those who w...",0.0000,Klein Blasts Congressman Over Government Shutd...,0.0000,Airline Faces Second Public Relations Disaster
6,319,Generate a headline for the following article:...,Tea Party Lawmakers' Reported Affair Cover-Up ...,0.0000,State Rep. Matt Rinaldi's call sparked an alte...,0.0000,Texas Lawmaker Calls ICE On Group Protesting A...,0.0870,State Rep Accused of Trying to Distract From A...
7,3110,Generate a headline for the following article:...,"ICE Officers Dined At Cafe, Then Arrested 3 Of...",0.0690,ICE Arrests: 'My family members were picked up...,0.0000,Immigration Enforcement Targets: Officials Cla...,0.0000,Immigration Officials Targeted Enforcement Act...
8,3111,Generate a headline for the following article:...,It’s Time For Presidential Candidates To Talk ...,0.0455,Activists Say Betsy DeVos' Nomination Puts Kid...,0.0000,Trump's Education Pick Faces Scrutiny Over Exp...,0.0000,Debate Ignored: Economic Issues Hard to Hear
9,3112,Generate a headline for the following article:...,"Detroit's Abandoned Ruins Are Captivating, But...",0.0000,'Dedicate Your Vote To A Woman' is the title f...,0.0952,Renovating the Past: How Cities Are Revitalizi...,0.0000,"Despite Vacant Land, Downtown Thrives Amidst H..."


### Mean Rouge-L by Model

,model,mean_rouge_l
1,lfm25MLX12B4bit,0.0514
2,qwen35MLX4B4bit,0.0292
0,flanT5Small,0.0142


...

Query : Generate a headline for the following article: The agents were “conducting a targeted enforcement action,” an immigration official said.
Gold  : ICE Officers Dined At Cafe, Then Arrested 3 Of Its Cooks, Owner Says
[flanT5Small] RL=0.0690
   ICE Arrests: 'My family members were picked up in the streets of Paris in the very same way'
[lfm25MLX12B4bit] RL=0.0000
   Immigration Enforcement Targets: Officials Claim Focused on Specific Individuals, Sparking Debate
[qwen35MLX4B4bit] RL=0.0000
   Immigration Officials Targeted Enforcement Action Leads to Scathing Testimony From Holocaust Survivor
----------------------------------------------------------------------------------------------------


#### Query 9 (qid=3111)

Query : Generate a headline for the following article: It's a key economic issue, but good luck hearing about it at a debate.
Gold  : It’s Time For Presidential Candidates To Talk About The New Housing Crisis
[flanT5Small] RL=0.0455
   Activists Say Betsy DeVos' Nomination Puts Kids' Civil Rights On The Line is the title for Trump's pick for Department of Education head faces questions about her experience at a confirmation hearing Tuesday
[lfm25MLX12B4bit] RL=0.0000
   Trump's Education Pick Faces Scrutiny Over Experience, Economic Concerns Dominate Debate
[qwen35MLX4B4bit] RL=0.0000
   Debate Ignored: Economic Issues Hard to Hear
----------------------------------------------------------------------------------------------------


#### Query 10 (qid=3112)

Query : Generate a headline for the following article: Despite large swaths of vacant land, many buildings are being developed, and the city's downtown is full of renovated historic
Gold  : Detroit's Abandoned Ruins Are Captivating, But Are They Bad For Neighborhoods?
[flanT5Small] RL=0.0000
   'Dedicate Your Vote To A Woman' is the title for Here are some of the many ways you can help nonprofit groups in your community., and Lawmakers Send Messages Of Support After Shots Fired At Congressmen is the title for House Majority Whip Steve Scalise is reportedly in stable condition after being shot in the hip., and 'Dedicate Your Vote To A Woman' Tweets Show Just How Far We’ve Come is the title for Here's to all the women who made this historic election possible., and Keith Ellison Says It
[lfm25MLX12B4bit] RL=0.0952
   Renovating the Past: How Cities Are Revitalizing Downtown Through Historic Development
[qwen35MLX4B4bit] RL=0.0000
   Despite Vacant Land, Downtown Thrives Amidst Historic Re